In [1]:
# Импорт
from ultralytics import YOLO
import torch
import yaml
import os
import shutil
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Блок 2: Проверка датасета
import yaml
import os
from pathlib import Path

# Путь к data.yaml (относительно корня проекта)
data_yaml_path = "../data/data.yaml"   # т.к. ноутбук в папке notebooks/

if not os.path.exists(data_yaml_path):
    # Альтернативный путь, если ноутбук запущен не из папки notebooks
    data_yaml_path = "data/data.yaml"

print(f"Используем data.yaml: {data_yaml_path}")

with open(data_yaml_path, 'r') as f:
    data_cfg = yaml.safe_load(f)

print("Содержимое data.yaml:")
for k, v in data_cfg.items():
    print(f"  {k}: {v}")

# Проверим существование путей к изображениям (можно вывести несколько примеров)
train_path = data_cfg.get('train')
val_path = data_cfg.get('val')
if train_path:
    print(f"\nПримеры тренировочных изображений: {train_path}")
    if os.path.exists(train_path):
        imgs = list(Path(train_path).glob("*.jpg"))[:3]
        for img in imgs:
            print(f"  {img}")
    else:
        print(f"  Внимание: путь {train_path} не найден. Возможно, нужно указать абсолютный путь или скорректировать data.yaml")

Используем data.yaml: ../data/data.yaml
Содержимое data.yaml:
  train: ../train/images
  val: ../valid/images
  test: ../test/images
  nc: 2
  names: ['cable tower', 'turbine']
  roboflow: {'workspace': 'kyle-graupe-jobhn', 'project': 'wind-farms', 'version': 5, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/kyle-graupe-jobhn/wind-farms/dataset/5'}

Примеры тренировочных изображений: ../train/images
  Внимание: путь ../train/images не найден. Возможно, нужно указать абсолютный путь или скорректировать data.yaml


In [3]:
# Блок 3: Исправление путей в data.yaml
import yaml
import os

data_yaml_path = "../data/data.yaml"
if not os.path.exists(data_yaml_path):
    data_yaml_path = "data/data.yaml"

with open(data_yaml_path, 'r') as f:
    data_cfg = yaml.safe_load(f)

# Изменяем пути: предполагаем, что папки train, valid, test находятся внутри data/
data_cfg['train'] = 'data/train/images'
data_cfg['val'] = 'data/valid/images'
data_cfg['test'] = 'data/test/images'

# Сохраняем обратно
with open(data_yaml_path, 'w') as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

print("Файл data.yaml обновлён. Новые пути:")
print(f"  train: {data_cfg['train']}")
print(f"  val: {data_cfg['val']}")
print(f"  test: {data_cfg['test']}")

Файл data.yaml обновлён. Новые пути:
  train: data/train/images
  val: data/valid/images
  test: data/test/images


In [4]:
# Проверка существования папок
from pathlib import Path
root = Path.cwd().parent  # предполагаем, что ноутбук в notebooks/, корень на уровень выше
for key in ['train', 'val', 'test']:
    p = root / data_cfg[key]
    print(f"{key}: {p} exists? {p.exists()}")

train: /Users/thegodaevfamily/ds_bootcamp/Meteors/Projects/CV_project/data/train/images exists? True
val: /Users/thegodaevfamily/ds_bootcamp/Meteors/Projects/CV_project/data/valid/images exists? True
test: /Users/thegodaevfamily/ds_bootcamp/Meteors/Projects/CV_project/data/test/images exists? True


In [8]:
# Блок 4: Перенос data.yaml в корень проекта и запуск обучения
import shutil
import os
import yaml
from pathlib import Path
from ultralytics import YOLO

# 1. Определим, где сейчас находится data.yaml и куда копировать
current_yaml = Path("../data/data.yaml").resolve()
if not current_yaml.exists():
    current_yaml = Path("data/data.yaml").resolve()

root_yaml = current_yaml.parent.parent / "data.yaml"   # корень проекта
print(f"Копируем {current_yaml} -> {root_yaml}")

# 2. Копируем файл
shutil.copy(current_yaml, root_yaml)

# 3. Загружаем скопированный файл и исправляем пути (они уже правильные, но проверим)
with open(root_yaml, 'r') as f:
    data_cfg = yaml.safe_load(f)

# Пути относительно корня проекта (где лежит data.yaml)
data_cfg['train'] = 'data/train/images'
data_cfg['val'] = 'data/valid/images'
data_cfg['test'] = 'data/test/images'

with open(root_yaml, 'w') as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

print("data.yaml перемещён в корень проекта и обновлён:")
print(f"  train: {data_cfg['train']}")
print(f"  val: {data_cfg['val']}")
print(f"  test: {data_cfg['test']}")

# 4. Обучение (для теста 10 эпох, потом увеличишь)
model = YOLO('yolo11n.pt')   # можно заменить на yolo11s.pt для лучшей точности

results = model.train(
    data=str(root_yaml),      # передаём полный путь к data.yaml в корне
    epochs=50,
    imgsz=440,
    batch=8,
    project='wind_train',
    name='exp',
    exist_ok=True,
    device='mps'              # или 'mps' если Mac M1/M2 и поддержка
)

print("Обучение успешно завершено!")

Копируем /Users/thegodaevfamily/ds_bootcamp/Meteors/Projects/CV_project/data/data.yaml -> /Users/thegodaevfamily/ds_bootcamp/Meteors/Projects/CV_project/data.yaml
data.yaml перемещён в корень проекта и обновлён:
  train: data/train/images
  val: data/valid/images
  test: data/test/images
Ultralytics 8.4.60 🚀 Python-3.13.7 torch-2.12.0 MPS (Apple M1)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/thegodaevfamily/ds_bootcamp/Meteors/Projects/CV_project/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, img

In [10]:
import os
from pathlib import Path

print("Текущая директория:", os.getcwd())
# Ищем wind_train начиная от корня проекта (предполагаем, что ноутбук в notebooks)
root = Path.cwd().parent
for p in root.rglob("wind_train"):
    print("Найдено:", p)

Текущая директория: /Users/thegodaevfamily/ds_bootcamp/Meteors/Projects/CV_project
Найдено: /Users/thegodaevfamily/ds_bootcamp/Meteors/Projects/CV_project/wind_train


In [11]:
train_exp = Path.cwd() / "wind_train" / "exp"   # если ноутбук в notebooks
if not train_exp.exists():
    # ищем рекурсивно
    for p in Path.cwd().rglob("wind_train/exp"):
        train_exp = p
        break
print("Папка с результатами:", train_exp)

Папка с результатами: /Users/thegodaevfamily/ds_bootcamp/Meteors/Projects/CV_project/wind_train/exp


In [14]:
from pathlib import Path
import shutil

# Корень проекта: поднимаемся на два уровня выше, если ноутбук в CV_PROJECT/notebooks/
# Или можно задать явно:
project_root = Path("/Users/thegodaevfamily/ds_bootcamp/Meteors/Projects/CV_PROJECT")

# Ищем все файлы best.pt в проекте
found_best = list(project_root.rglob("best.pt"))

if not found_best:
    print("❌ best.pt не найден. Проверьте, что обучение завершилось успешно.")
else:
    # Берём самый свежий (по времени изменения)
    best_pt = max(found_best, key=lambda p: p.stat().st_mtime)
    print(f"✅ Найден best.pt: {best_pt}")
    
    # Создаём папку models (если её нет)
    models_dir = project_root / "models"
    models_dir.mkdir(exist_ok=True)
    
    # Копируем с перезаписью
    target = models_dir / "yolo_wind.pt"
    shutil.copy2(best_pt, target)  # copy2 сохраняет метаданные
    print(f"✅ Веса скопированы в {target}")

# Дополнительно: скопируем графики, если они есть
images_dir = project_root / "images"
images_dir.mkdir(exist_ok=True)

# Список файлов, которые есть в папке exp (по скриншоту)
graphs = [
    ("BoxPR_curve.png", "wind_boxpr_curve.png"),
    ("confusion_matrix.png", "wind_confusion_matrix.png"),
    ("results.png", "wind_results.png"),
    ("BoxP_curve.png", "wind_boxp_curve.png"),
    ("BoxR_curve.png", "wind_boxr_curve.png"),
]

for src_name, dst_name in graphs:
    src = best_pt.parent.parent / src_name
    if src.exists():
        shutil.copy2(src, images_dir / dst_name)
        print(f"✅ Скопирован {src_name}")
    else:
        print(f"⚠️ График {src_name} не найден")

✅ Найден best.pt: /Users/thegodaevfamily/ds_bootcamp/Meteors/Projects/CV_PROJECT/wind_train/exp/weights/best.pt
✅ Веса скопированы в /Users/thegodaevfamily/ds_bootcamp/Meteors/Projects/CV_PROJECT/models/yolo_wind.pt
✅ Скопирован BoxPR_curve.png
✅ Скопирован confusion_matrix.png
✅ Скопирован results.png
✅ Скопирован BoxP_curve.png
✅ Скопирован BoxR_curve.png


In [15]:
# Удаляем старый data.yaml в data/, если он есть
old_yaml = Path.cwd().parent / "data" / "data.yaml"
if old_yaml.exists():
    old_yaml.unlink()
    print("Удалён data/data.yaml")

# Удаляем папку wind_train (опционально, можно оставить)
train_dir = Path.cwd() / "wind_train"
if train_dir.exists():
    shutil.rmtree(train_dir)
    print("Удалена временная папка wind_train")

Удалена временная папка wind_train
